In [7]:
# Drive access
from google.colab import drive, files, userdata
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [9]:
import os
import shutil

# Define your designated workspace folder for the IDP project
target_folder_name = input('Enter folder name (e.g., 01_IDP_Renal_Calculi): ')
target_folder = f"/content/drive/MyDrive/{target_folder_name}"

# Automatically create the folder if it doesn't exist
os.makedirs(target_folder, exist_ok=True)

# 1. Upload the files
print("Please upload your CSV files (e.g., herbal_drugs.csv and phytochemicals.csv):")
uploaded = files.upload()

# 2. Move the uploaded files into the target folder using copy and remove for robustness
for filename in uploaded.keys():
    destination_path = os.path.join(target_folder, filename)
    shutil.copy(filename, destination_path) # Copy the file to the destination
    os.remove(filename) # Remove the original file from the Colab VM
    print(f"Success: Copied '{filename}' to '{target_folder}' and removed original.")

print("\nAll files successfully uploaded and stored.")

Enter folder name (e.g., 01_IDP_Renal_Calculi): 02_Renal_calculi
Please upload your CSV files (e.g., herbal_drugs.csv and phytochemicals.csv):


Saving herbal_drugs.csv to herbal_drugs.csv
Saving phytochemicals.csv to phytochemicals.csv
Success: Copied 'herbal_drugs.csv' to '/content/drive/MyDrive/02_Renal_calculi' and removed original.
Success: Copied 'phytochemicals.csv' to '/content/drive/MyDrive/02_Renal_calculi' and removed original.

All files successfully uploaded and stored.


In [10]:
import pandas as pd
import numpy as np

# Define paths (assuming the standard names from our previous datasets)
path_herbs = os.path.join(target_folder, 'herbal_drugs.csv')
path_phyto = os.path.join(target_folder, 'phytochemicals.csv')

# Reading the uploaded files
df_herbs = pd.read_csv(path_herbs, skipinitialspace=True)
df_phyto = pd.read_csv(path_phyto, skipinitialspace=True)

print("--- Herbal Data Overview ---")
display(df_herbs.head(2))

print("\n--- Phytochemical Data Overview ---")
display(df_phyto.head(2))

--- Herbal Data Overview ---


,Plant_ID,Scientific_Name,Common_Name,Family,Part_Used,Traditional_Use
0,HD_001,Phyllanthus niruri,Chanca Piedra,Phyllanthaceae,Whole plant,Stone breaker; diuretic
1,HD_002,Boerhavia diffusa,Punarnava,Nyctaginaceae,Roots,Anti-inflammatory; lithotriptic



--- Phytochemical Data Overview ---


,Compound_ID,Scientific_Name,Phytochemical_Name,Compound_Class,Mechanism_of_Action,PubChem_CID
0,PC_101,Phyllanthus niruri,Phyllanthin,Lignan,Inhibits calcium oxalate crystal internalization,103130
1,PC_102,Phyllanthus niruri,Rutin,Flavonoid,Antioxidant; reduces oxidative stress in kidneys,5280805


In [11]:
# 1. Merge the Datasets
# Connecting the plant data to the chemical data using the Scientific_Name
df_merged = pd.merge(df_herbs, df_phyto, on='Scientific_Name', how='inner')

# 2. Text Standardization
# Strip trailing/leading white spaces from all string columns
text_cols = df_merged.select_dtypes(include=['object']).columns
for col in text_cols:
    df_merged[col] = df_merged[col].str.strip()

# 3. Handling Missing Values
# For pharmacological data, 'Unknown' or 'Not Specified' is safer than imputing medians
df_merged.fillna('Not Specified', inplace=True)

# 4. Removing Duplicates
# Drop rows where every single column is an exact duplicate
initial_shape = df_merged.shape
df_merged = df_merged.drop_duplicates()
print(f"Dropped {initial_shape[0] - df_merged.shape[0]} duplicate rows.")

# 5. Export Cleaned & Merged Data
output_filename = "cleaned_merged_renal_calculi.csv"
output_path = os.path.join(target_folder, output_filename)

df_merged.to_csv(output_path, index=False)

print(f"Final dataset shape: {df_merged.shape}")
print(f"Data cleaning complete. Cleaned file saved securely at: {output_path}")

Dropped 0 duplicate rows.
Final dataset shape: (7, 11)
Data cleaning complete. Cleaned file saved securely at: /content/drive/MyDrive/02_Renal_calculi/cleaned_merged_renal_calculi.csv


In [15]:
import subprocess
import os
from google.colab import userdata

# ==========================================
# Configuration: Update these variables
# ==========================================

FOLDER_PATH = "/content/drive/MyDrive/02_Renal_calculi"  # Updated to your actual path
GITHUB_USERNAME = "arinskyyyy"
REPO_NAME = "data-cleaning"
COMMIT_MESSAGE = "Update IDP renal calculi files via Colab"
USER_EMAIL = "arinskyyyy@gmail.com"
USER_NAME = "arinskyyyy"
BRANCH = "main"

# ==========================================

def run_command(command):
    """Helper function to run shell commands and print the output."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ {result.stdout.strip()}")
    else:
        print(f"❌ Error: {result.stderr.strip()}")

# 1. Retrieve the token securely from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("Could not find GITHUB_TOKEN. Please add it to Colab Secrets (the key icon on the left).")

# Construct the secure remote URL
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 2. Navigate to the target folder
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"The folder {FOLDER_PATH} does not exist.")
os.chdir(FOLDER_PATH)
print(f"Changed directory to: {os.getcwd()}")

# 3. Execute Git Commands
print("\nConfiguring Git...")
run_command("git init")
run_command(f'git config --global user.email "{USER_EMAIL}"')
run_command(f'git config --global user.name "{USER_NAME}"')

print("\nStaging and Committing...")
run_command("git add .")
# Added --allow-empty so it doesn't throw an error if no files were changed
run_command(f'git commit -m "{COMMIT_MESSAGE}" --allow-empty')

print("\nPushing to GitHub (Force Push)...")
run_command("git remote remove origin")
run_command(f"git remote add origin {REPO_URL}")
run_command(f"git branch -M {BRANCH}")

# Added --force to bypass the fetch rejection
run_command(f"git push -u origin {BRANCH} --force")

Changed directory to: /content/drive/MyDrive/02_Renal_calculi

Configuring Git...
✅ Reinitialized existing Git repository in /content/drive/MyDrive/02_Renal_calculi/.git/
✅ 
✅ 

Staging and Committing...
✅ 
✅ [main 123010b] Update IDP renal calculi files via Colab

Pushing to GitHub (Force Push)...
✅ 
✅ 
✅ 
✅ Branch 'main' set up to track remote branch 'main' from 'origin'.
